# 6. Construção da Camada Gold

## 6.1 Objetivo

A camada Gold representa a etapa analítica do Lakehouse, na qual os dados tratados e consolidados na camada Silver são organizados de acordo com o modelo dimensional definido para o projeto.

Nesta etapa, a tabela `workspace.silver.producao_mar` será utilizada como fonte para a construção de um modelo estrela composto pelas seguintes tabelas:

- `dim_tempo`;
- `dim_campo`;
- `dim_poco`;
- `dim_instalacao`;
- `fato_producao`.

As dimensões utilizarão chaves substitutas (*surrogate keys*), que serão posteriormente incorporadas à tabela fato como chaves estrangeiras.

A tabela `fato_producao` manterá a granularidade mensal por combinação de poço, campo e instalação identificada e validada durante a etapa de modelagem.

Para atender às perguntas de negócio, serão mantidas na tabela fato duas medidas analíticas:

- `producao_oleo_mmbbl`: produção de óleo expressa em milhões de barris (MMbbl);
- `producao_gas_bilhoes_m3`: produção total de gás natural expressa em bilhões de metros cúbicos, derivada da soma da produção de gás associado e gás não associado.

As conversões de unidade serão realizadas na camada Gold, preservando na camada Silver os valores nas unidades padronizadas a partir da fonte.

A construção será acompanhada por validações de unicidade das dimensões, integridade dos relacionamentos, quantidade de registros e cobertura temporal, garantindo que a transformação para o modelo dimensional não provoque perda indevida de informações.

## 6.2 Preparação da camada Gold

Antes da construção das tabelas dimensionais, será criado o schema `gold` no catálogo `workspace`.

A tabela consolidada da camada Silver será então carregada como fonte das transformações. Nesta etapa, será realizada apenas a preparação do ambiente, sem alteração dos dados armazenados na Silver.

In [0]:
# Cria o schema destinado às tabelas analíticas da camada Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# Carrega a tabela consolidada da camada Silver
df_silver = spark.table("workspace.silver.producao_mar")

# Valida o acesso à fonte e sua quantidade de registros
print(f"Total de registros na Silver: {df_silver.count():,}")
print(f"Total de colunas na Silver: {len(df_silver.columns)}")

Total de registros na Silver: 323,307
Total de colunas na Silver: 21


### 6.2.1 Resultado da preparação da camada Gold

A tabela `workspace.silver.producao_mar` foi carregada com sucesso como fonte para a construção da camada Gold.

A validação inicial retornou:

- **323.307 registros**;
- **21 colunas**.

Os valores correspondem à estrutura consolidada e previamente validada na camada Silver, confirmando que a fonte utilizada para a modelagem dimensional foi carregada integralmente.

O schema `workspace.gold` também foi preparado para receber as dimensões e a tabela fato que serão construídas nas próximas etapas.

## 6.3 Construção da dimensão Tempo

A dimensão `dim_tempo` será construída a partir dos valores distintos do atributo `mes_ano` presentes na camada Silver.

Como os dados possuem granularidade mensal, cada registro da dimensão representará um mês do período analisado. A dimensão será composta pelos seguintes atributos:

- `sk_tempo`: chave substituta da dimensão;
- `mes_ano`: data de referência do mês;
- `ano`: ano de referência;
- `mes`: número do mês;
- `nome_mes`: nome do mês.

A chave substituta será gerada de forma sequencial a partir da ordenação cronológica de `mes_ano`, permitindo que cada período possua um identificador único na camada Gold.

In [0]:
import warnings

# Suprime o aviso de performance gerado pelas janelas globais utilizadas
# na criação das surrogate keys das dimensões.
warnings.filterwarnings(
    "ignore",
    message=".*No Partition Defined for Window operation.*"
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define a ordenação cronológica utilizada para gerar a chave substituta
janela_tempo = Window.orderBy("mes_ano")

# Mapeia o número do mês para seu nome em português
nomes_meses = F.create_map(
    *[item for par in [
        (F.lit(1), F.lit("Janeiro")),
        (F.lit(2), F.lit("Fevereiro")),
        (F.lit(3), F.lit("Março")),
        (F.lit(4), F.lit("Abril")),
        (F.lit(5), F.lit("Maio")),
        (F.lit(6), F.lit("Junho")),
        (F.lit(7), F.lit("Julho")),
        (F.lit(8), F.lit("Agosto")),
        (F.lit(9), F.lit("Setembro")),
        (F.lit(10), F.lit("Outubro")),
        (F.lit(11), F.lit("Novembro")),
        (F.lit(12), F.lit("Dezembro"))
    ] for item in par]
)

# Constrói a dimensão Tempo a partir dos meses distintos da Silver
dim_tempo = (
    df_silver
    .select("mes_ano")
    .distinct()
    .orderBy("mes_ano")
    .withColumn(
        "sk_tempo",
        F.row_number().over(janela_tempo)
    )
    .withColumn("ano", F.year("mes_ano"))
    .withColumn("mes", F.month("mes_ano"))
    .withColumn("nome_mes", nomes_meses[F.col("mes")])
    .select(
        "sk_tempo",
        "mes_ano",
        "ano",
        "mes",
        "nome_mes"
    )
)

display(dim_tempo.limit(10))

sk_tempo,mes_ano,ano,mes,nome_mes
1,2005-01-01,2005,1,Janeiro
2,2005-02-01,2005,2,Fevereiro
3,2005-03-01,2005,3,Março
4,2005-04-01,2005,4,Abril
5,2005-05-01,2005,5,Maio
6,2005-06-01,2005,6,Junho
7,2005-07-01,2005,7,Julho
8,2005-08-01,2005,8,Agosto
9,2005-09-01,2005,9,Setembro
10,2005-10-01,2005,10,Outubro


In [0]:
dim_tempo.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("sk_tempo").alias("sk_distintas"),
    F.countDistinct("mes_ano").alias("meses_distintos"),
    F.min("mes_ano").alias("mes_inicial"),
    F.max("mes_ano").alias("mes_final")
).display()

dim_tempo.printSchema()

total_registros,sk_distintas,meses_distintos,mes_inicial,mes_final
252,252,252,2005-01-01,2025-12-01


root
 |-- sk_tempo: integer (nullable = false)
 |-- mes_ano: date (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- nome_mes: string (nullable = true)



In [0]:
# Verifica a existência de valores nulos nos atributos da dimensão Tempo
dim_tempo.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}")
        for c in dim_tempo.columns
    ]
).display()

nulos_sk_tempo,nulos_mes_ano,nulos_ano,nulos_mes,nulos_nome_mes
0,0,0,0,0


### 6.3.1 Resultado da construção e validação da dimensão Tempo

A dimensão `dim_tempo` foi construída a partir dos períodos mensais distintos existentes na camada Silver.

A validação apresentou os seguintes resultados:

- **252 registros** na dimensão;
- **252 chaves substitutas distintas**;
- **252 períodos mensais distintos**;
- período inicial em **janeiro de 2005**;
- período final em **dezembro de 2025**;
- **nenhum valor nulo** nos atributos da dimensão.

Os resultados confirmam a unicidade da chave substituta e a correspondência de um registro para cada mês existente no período analisado. Os 252 períodos representam os 21 anos completos compreendidos entre 2005 e 2025.

Embora alguns atributos sejam apresentados pelo schema do DataFrame como `nullable = true`, a validação dos dados demonstrou que não existem valores nulos nessas colunas.

Dessa forma, a dimensão Tempo encontra-se consistente e apta para persistência na camada Gold.

### 6.3.2 Persistência da dimensão Tempo

Após a validação de sua estrutura, unicidade e completude, a dimensão Tempo será persistida no schema `workspace.gold` em formato Delta.

A persistência materializa a primeira tabela do modelo dimensional definido para a camada Gold.

In [0]:
# Persiste a dimensão Tempo como tabela Delta na camada Gold
(
    dim_tempo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_tempo")
)

# Confirma a quantidade de registros persistidos
spark.table("workspace.gold.dim_tempo").count()

252

### 6.3.3 Resultado da persistência da dimensão Tempo

A dimensão `dim_tempo` foi persistida com sucesso como tabela Delta em `workspace.gold.dim_tempo`.

Após a gravação, a tabela apresentou **252 registros**, mantendo a mesma quantidade previamente validada no DataFrame utilizado para sua construção.

Com isso, a dimensão Tempo encontra-se concluída na camada Gold.

## 6.4 Construção da dimensão Campo

A dimensão `dim_campo` será construída a partir dos atributos `campo`, `bacia` e `estado` existentes na camada Silver.

As análises realizadas durante a modelagem demonstraram que cada campo está associado a uma única bacia. Para o atributo estado, foram identificados três campos associados tanto a um estado identificado quanto à categoria `Não Informado`.

Para evitar a criação de mais de um registro dimensional para o mesmo campo, será aplicada a regra definida durante a modelagem:

> quando um campo possuir um estado identificado, esse estado será utilizado na dimensão; a categoria `Não Informado` será mantida apenas quando não existir um estado identificado para o campo.

Essa consolidação será realizada exclusivamente na camada Gold e não altera os registros preservados na camada Silver.

Após a definição dos atributos de cada campo, será gerada a chave substituta `sk_campo`, formando a chave primária da dimensão.

In [0]:
# Define a janela utilizada para consolidar os atributos de cada campo
janela_campo = Window.partitionBy("campo")

# Constrói uma base com um único estado representativo para cada campo
base_campo = (
    df_silver
    .select("campo", "bacia", "estado")
    .distinct()
    .withColumn(
        "estado_identificado",
        F.first(
            F.when(
                (F.col("estado").isNotNull()) &
                (F.col("estado") != "Não Informado"),
                F.col("estado")
            ),
            ignorenulls=True
        ).over(janela_campo)
    )
    .withColumn(
        "estado_final",
        F.coalesce(
            F.col("estado_identificado"),
            F.lit("Não Informado")
        )
    )
    .select(
        "campo",
        "bacia",
        F.col("estado_final").alias("estado")
    )
    .distinct()
)

# Gera uma chave substituta sequencial para cada campo
janela_sk_campo = Window.orderBy("campo")

dim_campo = (
    base_campo
    .withColumn(
        "sk_campo",
        F.row_number().over(janela_sk_campo)
    )
    .select(
        "sk_campo",
        "campo",
        "bacia",
        "estado"
    )
)

display(dim_campo.limit(10))

sk_campo,campo,bacia,estado
1,ABALONE,Campos,Espírito Santo
2,AGULHA,Potiguar,Rio Grande do Norte
3,ALBACORA,Campos,Rio de Janeiro
4,ALBACORA LESTE,Campos,Rio de Janeiro
5,ANC_BRAVA,Campos,Rio de Janeiro
6,ANC_LULA,Santos,Rio de Janeiro
7,ANC_NORTE_ATAPU,Santos,Rio de Janeiro
8,ANC_TARTARUGA VERDE,Campos,Rio de Janeiro
9,ANC_TUPI,Santos,Rio de Janeiro
10,ANEQUIM,Campos,Rio de Janeiro


### 6.4.1 Validação da dimensão Campo

Após a construção da dimensão, serão verificadas sua unicidade e completude, além da consistência das relações entre campo, bacia e estado.

A validação busca confirmar que:

- cada campo possui apenas um registro na dimensão;
- cada chave substituta é única;
- não existem valores nulos nos atributos dimensionais;
- a regra de consolidação dos estados resultou em uma única associação de estado por campo.

In [0]:
# Valida quantidade de registros, unicidade e completude da dimensão Campo
validacao_dim_campo = dim_campo.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("sk_campo").alias("sk_distintas"),
    F.countDistinct("campo").alias("campos_distintos"),
    F.sum(F.col("sk_campo").isNull().cast("int")).alias("nulos_sk_campo"),
    F.sum(F.col("campo").isNull().cast("int")).alias("nulos_campo"),
    F.sum(F.col("bacia").isNull().cast("int")).alias("nulos_bacia"),
    F.sum(F.col("estado").isNull().cast("int")).alias("nulos_estado")
)

display(validacao_dim_campo)

# Verifica se algum campo ainda aparece mais de uma vez na dimensão
campos_duplicados = (
    dim_campo
    .groupBy("campo")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Campos com mais de um registro na dimensão: {campos_duplicados.count()}")

total_registros,sk_distintas,campos_distintos,nulos_sk_campo,nulos_campo,nulos_bacia,nulos_estado
144,144,144,0,0,0,0


Campos com mais de um registro na dimensão: 0


### 6.4.2 Resultado da validação da dimensão Campo

A dimensão `dim_campo` foi construída e validada após as padronizações realizadas na camada Silver.

A validação apresentou:

- **144 registros**;
- **144 chaves substitutas distintas**;
- **144 campos distintos**;
- **nenhum campo com mais de um registro na dimensão**;
- **nenhum valor nulo** nos atributos `sk_campo`, `campo`, `bacia` e `estado`.

Os resultados confirmam que cada campo está representado uma única vez na dimensão e associado a uma única chave substituta.

A regra definida para o atributo estado também permitiu manter um único estado representativo por campo, priorizando a informação identificada em relação à categoria `Não Informado`.

Dessa forma, a dimensão Campo encontra-se consistente e apta para persistência na camada Gold.

### 6.4.3 Persistência da dimensão Campo

Após a validação de sua unicidade e completude, a dimensão Campo será persistida no schema `workspace.gold` em formato Delta.

In [0]:
# Persiste a dimensão Campo como tabela Delta na camada Gold
(
    dim_campo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_campo")
)

# Confirma a quantidade de registros persistidos
spark.table("workspace.gold.dim_campo").count()

144

### 6.4.4 Resultado da persistência da dimensão Campo

A dimensão `dim_campo` foi persistida com sucesso como tabela Delta em `workspace.gold.dim_campo`.

Após a gravação, foram confirmados **144 registros**, mantendo a mesma quantidade obtida e validada durante a construção da dimensão.

Com isso, a dimensão Campo encontra-se concluída e disponível na camada Gold para posterior relacionamento com a tabela fato.

## 6.5 Construção da dimensão Poço

A dimensão `dim_poco` será construída a partir dos valores distintos do atributo `poco` presentes na camada Silver.

Cada poço será representado uma única vez na dimensão e receberá uma chave substituta `sk_poco`, que posteriormente será utilizada como chave estrangeira na tabela fato.

Como a análise realizada durante a modelagem não identificou valores nulos no atributo `poco`, não será necessária a criação de uma categoria para informações ausentes.

In [0]:
# Define a ordenação utilizada para geração das chaves substitutas
janela_sk_poco = Window.orderBy("poco")

# Constrói a dimensão Poço
dim_poco = (
    df_silver
    .select("poco")
    .distinct()
    .withColumn(
        "sk_poco",
        F.row_number().over(janela_sk_poco)
    )
    .select(
        "sk_poco",
        "poco"
    )
)

display(dim_poco.limit(10))

sk_poco,poco
1,1-BRSA-108A-ESS
2,1-BRSA-1116-RJS
3,1-BRSA-1146-RJS
4,1-BRSA-319-RJS
5,1-BRSA-594-SPS
6,1-BRSA-607-SPS
7,1-BRSA-658-SPS
8,1-BRSA-713-RJS
9,1-BRSA-976-RJS
10,1-ESS-77-ESS


### 6.5.1 Validação da dimensão Poço

Após a construção da dimensão, serão verificadas a quantidade de registros, a unicidade das chaves substitutas e dos poços e a existência de valores nulos.

In [0]:
dim_poco.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("sk_poco").alias("sk_distintas"),
    F.countDistinct("poco").alias("pocos_distintos"),
    F.sum(F.col("sk_poco").isNull().cast("int")).alias("nulos_sk_poco"),
    F.sum(F.col("poco").isNull().cast("int")).alias("nulos_poco")
).display()

total_registros,sk_distintas,pocos_distintos,nulos_sk_poco,nulos_poco
2503,2503,2503,0,0


### 6.5.2 Resultado da validação da dimensão Poço

A validação da dimensão `dim_poco` apresentou:

- **2.503 registros**;
- **2.503 chaves substitutas distintas**;
- **2.503 poços distintos**;
- **nenhum valor nulo** nos atributos da dimensão.

Os resultados confirmam que cada poço está representado uma única vez e associado a uma única chave substituta.

Dessa forma, a dimensão Poço encontra-se consistente e apta para persistência na camada Gold.

### 6.5.3 Persistência da dimensão Poço

Após a validação de sua unicidade e completude, a dimensão Poço será persistida no schema `workspace.gold` em formato Delta.

In [0]:
# Persiste a dimensão Poço como tabela Delta na camada Gold
(
    dim_poco.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_poco")
)

# Confirma a quantidade de registros persistidos
spark.table("workspace.gold.dim_poco").count()

2503

### 6.5.4 Resultado da persistência da dimensão Poço

A dimensão `dim_poco` foi persistida com sucesso como tabela Delta em `workspace.gold.dim_poco`.

Após a gravação, foram confirmados **2.503 registros**, mantendo a mesma quantidade previamente validada.

Com isso, a dimensão Poço encontra-se concluída e disponível na camada Gold para relacionamento com a tabela fato.

## 6.6 Construção da dimensão Instalação

A dimensão `dim_instalacao` será construída a partir dos valores distintos do atributo `instalacao` presentes na camada Silver.

Durante a análise dos dados, foram identificados registros em que a instalação não foi informada. Para permitir que todos os registros da futura tabela fato possuam uma chave correspondente na dimensão, os valores nulos serão associados à categoria `Não Informado` durante a construção da camada Gold.

Essa transformação não modifica os dados preservados na camada Silver, sendo aplicada apenas para viabilizar a integridade dos relacionamentos do modelo dimensional.

Cada instalação será representada uma única vez e receberá uma chave substituta `sk_instalacao`.

In [0]:
# Substitui valores nulos pela categoria dimensional "Não Informado"
base_instalacao = (
    df_silver
    .select(
        F.coalesce(
            F.col("instalacao"),
            F.lit("Não Informado")
        ).alias("instalacao")
    )
    .distinct()
)

# Define a ordenação para geração das chaves substitutas
janela_sk_instalacao = Window.orderBy("instalacao")

# Constrói a dimensão Instalação
dim_instalacao = (
    base_instalacao
    .withColumn(
        "sk_instalacao",
        F.row_number().over(janela_sk_instalacao)
    )
    .select(
        "sk_instalacao",
        "instalacao"
    )
)

display(dim_instalacao.limit(10))

sk_instalacao,instalacao
1,ALPHA STAR
2,AMARALINA STAR
3,ATLANTIC STAR
4,BRAVA STAR
5,CAROLINA
6,DEEPWATER AQUILA
7,DEEPWATER NAVIGATOR
8,DYNAMIC PRODUCER
9,Deepwater Corcovado
10,ENSCO 6003


### 6.6.1 Validação da dimensão Instalação

Após a construção da dimensão `dim_instalacao`, serão verificadas a quantidade de registros, a unicidade das chaves substitutas e das instalações e a existência de valores nulos.

Também será confirmada a presença da categoria `Não Informado`, utilizada na camada Gold para representar os registros cuja instalação não foi identificada na camada Silver.

Essa validação permite verificar se cada instalação está representada uma única vez e se a dimensão está adequada para o posterior relacionamento com a tabela fato.

In [0]:
dim_instalacao.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("sk_instalacao").alias("sk_distintas"),
    F.countDistinct("instalacao").alias("instalacoes_distintas"),
    F.sum(F.col("sk_instalacao").isNull().cast("int")).alias("nulos_sk_instalacao"),
    F.sum(F.col("instalacao").isNull().cast("int")).alias("nulos_instalacao")
).display()

# Confirma a existência da categoria utilizada para instalações não identificadas
dim_instalacao.filter(
    F.col("instalacao") == "Não Informado"
).display()

total_registros,sk_distintas,instalacoes_distintas,nulos_sk_instalacao,nulos_instalacao
231,231,231,0,0


sk_instalacao,instalacao
75,Não Informado


### 6.6.2 Resultado da validação da dimensão Instalação

A validação da dimensão `dim_instalacao` apresentou:

- **231 registros**;
- **231 chaves substitutas distintas**;
- **231 instalações distintas**;
- **nenhum valor nulo** nos atributos da dimensão;
- presença da categoria `Não Informado`, associada à chave substituta `sk_instalacao = 75`.

Os resultados confirmam que cada instalação está representada uma única vez e associada a uma única chave substituta.

A categoria `Não Informado` permitirá que os registros cuja instalação não foi identificada na camada Silver sejam relacionados à dimensão sem gerar chaves estrangeiras nulas na futura tabela fato.

Dessa forma, a dimensão Instalação encontra-se consistente e apta para persistência na camada Gold.

### 6.6.3 Persistência da dimensão Instalação

Após a validação de sua unicidade e completude, a dimensão Instalação será persistida no schema `workspace.gold` em formato Delta.

In [0]:
# Persiste a dimensão Instalação como tabela Delta na camada Gold
(
    dim_instalacao.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_instalacao")
)

# Confirma a quantidade de registros persistidos
spark.table("workspace.gold.dim_instalacao").count()

231

### 6.6.4 Resultado da persistência da dimensão Instalação

A dimensão `dim_instalacao` foi persistida com sucesso como tabela Delta em `workspace.gold.dim_instalacao`.

Após a gravação, foram confirmados **231 registros**, mantendo a mesma quantidade previamente validada.

Com isso, a dimensão Instalação encontra-se concluída e disponível na camada Gold para relacionamento com a tabela fato.

## 6.7 Construção da tabela fato de produção

A tabela `fato_producao` representa as observações mensais de produção e mantém a granularidade definida durante a modelagem: uma observação por combinação de período, campo, poço e instalação.

A tabela será relacionada às quatro dimensões por meio de suas respectivas chaves substitutas e conterá duas medidas destinadas ao consumo analítico:

- `producao_oleo_mmbbl`: produção de óleo expressa em milhões de barris (MMbbl);
- `producao_gas_bilhoes_m3`: produção total de gás natural expressa em bilhões de metros cúbicos.

As medidas são derivadas dos valores preservados na camada Silver. A produção de óleo será convertida de metros cúbicos para milhões de barris utilizando o fator de conversão de **1 m³ = 6,28981 barris**.

Para o gás natural, será inicialmente calculada a soma da produção de gás associado e não associado. Como os valores de origem são expressos em milhares de metros cúbicos, o resultado será convertido para bilhões de metros cúbicos.

A conversão das unidades é realizada apenas na camada Gold, destinada ao consumo analítico. A camada Silver permanece preservando os valores e unidades padronizados a partir da fonte.

### 6.7.1 Avaliação dos valores nulos na produção de gás

Antes da construção da tabela fato, será avaliado o comportamento dos valores nulos nos atributos `producao_gas_associado_mm3` e `producao_gas_nao_associado_mm3`.

Embora a nomenclatura das colunas tenha sido preservada a partir da representação utilizada na fonte, os metadados da ANP indicam que os volumes de gás correspondem a **milhares de metros cúbicos**.

Essa verificação é necessária porque a medida analítica de produção total de gás será derivada pela soma das produções de gás associado e não associado e, posteriormente, convertida para bilhões de metros cúbicos na camada Gold.

Serão identificadas as ocorrências em que ambas as parcelas possuem valor, apenas uma delas possui valor ou ambas são nulas. O resultado permitirá definir a regra de cálculo da produção total de gás sem atribuir aos valores ausentes um significado não sustentado pelos dados.

In [0]:
# Avalia as combinações de nulidade das duas parcelas de produção de gás
validacao_gas = (
    df_silver
    .select(
        F.when(
            F.col("producao_gas_associado_mm3").isNotNull() &
            F.col("producao_gas_nao_associado_mm3").isNotNull(),
            "Ambos preenchidos"
        )
        .when(
            F.col("producao_gas_associado_mm3").isNotNull() &
            F.col("producao_gas_nao_associado_mm3").isNull(),
            "Somente gás associado preenchido"
        )
        .when(
            F.col("producao_gas_associado_mm3").isNull() &
            F.col("producao_gas_nao_associado_mm3").isNotNull(),
            "Somente gás não associado preenchido"
        )
        .otherwise("Ambos nulos")
        .alias("situacao")
    )
    .groupBy("situacao")
    .count()
    .orderBy("situacao")
)

display(validacao_gas)

situacao,count
Ambos nulos,74957
Ambos preenchidos,248350


#### 6.7.1.1 Resultado da avaliação dos valores nulos

A avaliação identificou dois padrões nos atributos utilizados para o cálculo da produção total de gás:

- **248.350 registros** possuem valores preenchidos tanto para gás associado quanto para gás não associado;
- **74.957 registros** possuem ambas as parcelas nulas.

Não foram identificados registros em que apenas uma das duas parcelas estivesse preenchida.

Dessa forma, a produção total de gás poderá ser calculada pela soma direta de `producao_gas_associado_mm3` e `producao_gas_nao_associado_mm3`. Nos registros em que ambas as parcelas são nulas, o resultado da soma permanecerá nulo, preservando a ausência de informação existente na fonte em vez de interpretá-la como produção igual a zero.

Após a soma das duas parcelas, o volume será convertido para **bilhões de metros cúbicos**, unidade definida para consumo analítico na camada Gold.

### 6.7.2 Preparação das medidas analíticas

Com base na avaliação dos valores nulos, a produção total de gás natural será calculada pela soma das produções de gás associado e não associado. Como não foram identificados registros em que apenas uma das duas parcelas estivesse preenchida, a soma direta preserva adequadamente os casos em que ambas possuem valor e mantém como nulos os registros em que ambas estão ausentes.

Para adequar as medidas ao consumo analítico da camada Gold, também serão realizadas conversões de unidade:

- a produção de óleo será convertida de metros cúbicos (m³) para **milhões de barris (MMbbl)**, utilizando o fator de conversão de 1 m³ = 6,28981 barris;
- a produção total de gás natural será convertida da unidade utilizada pela fonte, correspondente a milhares de metros cúbicos, para **bilhões de metros cúbicos**.

As conversões serão realizadas somente na camada Gold. Os valores nas unidades provenientes da fonte permanecem preservados na camada Silver.

In [0]:
# Prepara as medidas da fato nas unidades definidas para consumo analítico.
df_fato_base = (
    df_silver
    .withColumn(
        "producao_oleo_mmbbl",
        (F.col("producao_oleo_m3") * F.lit(6.28981)) / F.lit(1_000_000)
    )
    .withColumn(
        "producao_gas_bilhoes_m3",
        (
            F.col("producao_gas_associado_mm3")
            + F.col("producao_gas_nao_associado_mm3")
        ) / F.lit(1_000_000)
    )
)

# Valida a criação das medidas analíticas e a preservação dos valores nulos.
display(
    df_fato_base.agg(
        F.count("*").alias("total_registros"),
        F.count("producao_oleo_mmbbl").alias("oleo_preenchido"),
        F.sum(F.col("producao_oleo_mmbbl").isNull().cast("int")).alias("oleo_nulo"),
        F.count("producao_gas_bilhoes_m3").alias("gas_preenchido"),
        F.sum(F.col("producao_gas_bilhoes_m3").isNull().cast("int")).alias("gas_nulo")
    )
)

total_registros,oleo_preenchido,oleo_nulo,gas_preenchido,gas_nulo
323307,248350,74957,248350,74957


#### 6.7.2.1 Resultado da preparação das medidas analíticas

A preparação das medidas analíticas manteve os **323.307 registros** existentes na camada Silver.

Foram construídas duas medidas destinadas à tabela fato:

- `producao_oleo_mmbbl`, obtida pela conversão da produção de óleo de metros cúbicos para **milhões de barris (MMbbl)**;
- `producao_gas_bilhoes_m3`, obtida pela soma das produções de gás associado e não associado e posterior conversão para **bilhões de metros cúbicos**.

A conversão das unidades não altera o tratamento dos valores ausentes. Para a produção de gás, foram mantidos **248.350 registros preenchidos** e **74.957 registros nulos**, reproduzindo o padrão identificado anteriormente nas parcelas utilizadas no cálculo.

Da mesma forma, valores nulos existentes na produção de óleo permanecem nulos após a conversão, sem substituição por zero.

Dessa forma, as duas medidas encontram-se preparadas para composição da tabela fato nas unidades definidas para consumo analítico da camada Gold.

### 6.7.3 Relacionamento com as dimensões

Após a preparação das medidas analíticas, os registros serão relacionados às dimensões Tempo, Campo, Poço e Instalação para obtenção das respectivas chaves substitutas.

Os relacionamentos serão realizados a partir dos atributos correspondentes da camada Silver. Para os registros sem instalação informada, será utilizada a categoria `Não Informado`, criada na dimensão Instalação para preservar esses registros no modelo dimensional.

Ao final, a tabela fato será composta exclusivamente pelas quatro chaves estrangeiras e pelas duas medidas analíticas definidas para a camada Gold.

In [0]:
# Prepara a instalação para o relacionamento com a dimensão,
# associando valores nulos à categoria "Não Informado"
df_fato_base = (
    df_fato_base
    .withColumn(
        "instalacao_gold",
        F.coalesce(
            F.col("instalacao"),
            F.lit("Não Informado")
        )
    )
)

# Relaciona os registros de produção às quatro dimensões
fato_producao = (
    df_fato_base.alias("f")
    
    .join(
        dim_tempo.alias("t"),
        F.col("f.mes_ano") == F.col("t.mes_ano"),
        "left"
    )
    
    .join(
        dim_campo.alias("c"),
        F.col("f.campo") == F.col("c.campo"),
        "left"
    )
    
    .join(
        dim_poco.alias("p"),
        F.col("f.poco") == F.col("p.poco"),
        "left"
    )
    
    .join(
        dim_instalacao.alias("i"),
        F.col("f.instalacao_gold") == F.col("i.instalacao"),
        "left"
    )
    
    .select(
    F.col("t.sk_tempo"),
    F.col("c.sk_campo"),
    F.col("p.sk_poco"),
    F.col("i.sk_instalacao"),
    F.col("f.producao_oleo_mmbbl"),
    F.col("f.producao_gas_bilhoes_m3")
    )
)

display(fato_producao.limit(10))

sk_tempo,sk_campo,sk_poco,sk_instalacao,producao_oleo_mmbbl,producao_gas_bilhoes_m3
251,3,350,54,0.0,0.0
251,3,370,54,0.0,0.0
246,3,350,54,0.0,0.0
246,3,368,54,0.0,0.0
246,3,370,54,0.0,0.0
243,3,369,54,0.0,0.0
243,3,350,54,0.0,0.0
243,3,368,54,0.0,0.0
243,3,370,54,0.0,0.0
245,3,369,54,0.0,0.0


### 6.7.4 Validação dos relacionamentos dimensionais

Após o relacionamento com as dimensões, será verificada a integridade das chaves estrangeiras da tabela fato.

A validação irá confirmar se a quantidade original de registros foi preservada e se todos os registros encontraram correspondência nas dimensões Tempo, Campo, Poço e Instalação.

A ausência de chaves estrangeiras nulas indicará que os relacionamentos dimensionais foram realizados de forma completa.

In [0]:
# Valida a integridade dos relacionamentos da tabela fato,
# verificando o total de registros e a existência de chaves estrangeiras nulas.
fato_producao.select(
    F.count("*").alias("total_registros"),
    
    F.sum(
        F.col("sk_tempo").isNull().cast("int")
    ).alias("fk_tempo_nulas"),
    
    F.sum(
        F.col("sk_campo").isNull().cast("int")
    ).alias("fk_campo_nulas"),
    
    F.sum(
        F.col("sk_poco").isNull().cast("int")
    ).alias("fk_poco_nulas"),
    
    F.sum(
        F.col("sk_instalacao").isNull().cast("int")
    ).alias("fk_instalacao_nulas")
).display()

total_registros,fk_tempo_nulas,fk_campo_nulas,fk_poco_nulas,fk_instalacao_nulas
323307,0,0,0,0


#### 6.7.4.1 Resultado da validação dos relacionamentos dimensionais

A validação dos relacionamentos dimensionais confirmou a presença de **323.307 registros** na tabela fato, mesma quantidade existente na camada Silver.

Também não foram identificadas chaves estrangeiras nulas:

- `sk_tempo`: **0** valores nulos;
- `sk_campo`: **0** valores nulos;
- `sk_poco`: **0** valores nulos;
- `sk_instalacao`: **0** valores nulos.

Os resultados demonstram que todos os registros encontraram correspondência nas quatro dimensões e que os relacionamentos não provocaram perda de registros.

Dessa forma, a integridade dos relacionamentos dimensionais foi confirmada, permitindo prosseguir para a validação da granularidade da tabela fato.

### 6.7.5 Validação da granularidade da tabela fato

Após a associação das chaves substitutas, será validada a granularidade definida para a tabela `fato_producao`.

Será verificado se existe mais de um registro para a mesma combinação de `sk_tempo`, `sk_campo`, `sk_poco` e `sk_instalacao`, que representam respectivamente o período, o campo, o poço e a instalação.

Essa verificação é especialmente importante após as padronizações realizadas nos atributos categóricos da camada Silver, pois permite confirmar que a consolidação de diferentes grafias não gerou duplicidades na granularidade definida para a tabela fato.

In [0]:
# Verifica se existem combinações repetidas das quatro
# chaves que definem a granularidade da tabela fato.
validacao_granularidade = (
    fato_producao
    .groupBy(
        "sk_tempo",
        "sk_campo",
        "sk_poco",
        "sk_instalacao"
    )
    .count()
)

validacao_granularidade.select(
    F.sum(
        (F.col("count") > 1).cast("int")
    ).alias("combinacoes_repetidas"),
    
    F.max("count").alias("max_registros_por_combinacao")
).display()

combinacoes_repetidas,max_registros_por_combinacao
0,1


### 6.7.5.1 Resultado da validação da granularidade

A validação não identificou combinações repetidas entre as chaves `sk_tempo`, `sk_campo`, `sk_poco` e `sk_instalacao`.

O número máximo de registros encontrado para uma mesma combinação de chaves foi **1**, confirmando a unicidade das observações na granularidade definida para a tabela fato.

Esse resultado demonstra que as padronizações realizadas anteriormente não provocaram a consolidação indevida de registros distintos.

Dessa forma, permanece válida a granularidade definida para `fato_producao`: uma observação mensal de produção por combinação de poço, campo e instalação.

### 6.7.6 Validação final da tabela Fato Produção

Antes da persistência, será realizada uma validação final da tabela `fato_producao`.

Serão verificados a quantidade total de registros, o período temporal coberto e a quantidade de valores nulos nas duas medidas analíticas: produção de óleo em **milhões de barris (MMbbl)** e produção total de gás natural em **bilhões de metros cúbicos**.

Essa etapa permite confirmar que a tabela fato mantém a cobertura histórica esperada e documentar o nível de preenchimento das medidas antes de sua gravação na camada Gold.

In [0]:
# Valida a quantidade de registros, a cobertura temporal
# e o preenchimento das medidas da tabela fato.
validacao_final_fato = (
    fato_producao.alias("f")
    .join(
        dim_tempo.select("sk_tempo", "mes_ano").alias("t"),
        "sk_tempo",
        "left"
    )
)

validacao_final_fato.select(
    F.count("*").alias("total_registros"),
    F.min("mes_ano").alias("mes_inicial"),
    F.max("mes_ano").alias("mes_final"),

    F.sum(
        F.col("producao_oleo_mmbbl").isNull().cast("int")
    ).alias("producao_oleo_nulos"),

    F.sum(
        F.col("producao_gas_bilhoes_m3").isNull().cast("int")
    ).alias("producao_gas_nulos")
).display()

total_registros,mes_inicial,mes_final,producao_oleo_nulos,producao_gas_nulos
323307,2005-01-01,2025-12-01,74957,74957


#### 6.7.6.1 Resultado da validação final da tabela Fato Produção

A validação final confirmou que a tabela `fato_producao` contém **323.307 registros**, cobrindo o período de **janeiro de 2005 a dezembro de 2025**.

Foram identificados **74.957 valores nulos** tanto na medida `producao_oleo_mmbbl` quanto na medida `producao_gas_bilhoes_m3`, mantendo o padrão de ausência de informação identificado anteriormente na camada Silver.

As conversões de unidade realizadas para consumo analítico não alteraram a quantidade de registros nem o comportamento dos valores nulos.

Dessa forma, a tabela fato mantém a cobertura histórica esperada e encontra-se preparada para persistência na camada Gold.

### 6.7.7 Persistência da tabela Fato Produção

Após a conclusão das validações de integridade, granularidade, cobertura temporal e preenchimento das medidas, a tabela `fato_producao` será persistida no schema `workspace.gold` em formato Delta.

A persistência conclui a implementação física do modelo dimensional definido para a camada Gold.

In [0]:
# Persiste a tabela fato em formato Delta, atualizando também seu schema.
(
    fato_producao.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("workspace.gold.fato_producao")
)

# Confirma a quantidade de registros persistidos.
spark.table("workspace.gold.fato_producao").count()

323307

### 6.7.7.1 Resultado da persistência da tabela Fato Produção

A tabela `fato_producao` foi persistida com sucesso como tabela Delta em `workspace.gold.fato_producao`.

Após a gravação, foram confirmados **323.307 registros**, mantendo a mesma quantidade validada antes da persistência.

Com isso, a tabela fato encontra-se concluída e disponível na camada Gold, juntamente com as quatro dimensões que compõem o modelo dimensional.

## 6.8 Validação final da camada Gold

Após a construção e persistência do modelo dimensional, será realizada uma validação consolidada das tabelas que compõem a camada Gold.

A verificação tem como objetivo confirmar a existência das quatro dimensões e da tabela fato e registrar a quantidade final de registros armazenados em cada uma delas.

In [0]:
# Lista as tabelas que compõem o modelo dimensional da camada Gold.
tabelas_gold = [
    "dim_tempo",
    "dim_campo",
    "dim_poco",
    "dim_instalacao",
    "fato_producao"
]

# Obtém a quantidade de registros persistidos em cada tabela.
resultado_gold = []

for tabela in tabelas_gold:
    quantidade = spark.table(f"workspace.gold.{tabela}").count()
    resultado_gold.append((tabela, quantidade))

df_validacao_gold = spark.createDataFrame(
    resultado_gold,
    ["tabela", "total_registros"]
)

display(df_validacao_gold)

tabela,total_registros
dim_tempo,252
dim_campo,144
dim_poco,2503
dim_instalacao,231
fato_producao,323307


### 6.8.1 Resultado da validação final da camada Gold

A validação consolidada confirmou a persistência das cinco tabelas que compõem o modelo dimensional da camada Gold:

- `dim_tempo`: **252 registros**;
- `dim_campo`: **144 registros**;
- `dim_poco`: **2.503 registros**;
- `dim_instalacao`: **231 registros**;
- `fato_producao`: **323.307 registros**.

Os resultados são consistentes com as validações realizadas individualmente durante a construção de cada tabela.

A tabela fato preservou a quantidade total de registros da camada Silver e encontra-se relacionada às quatro dimensões por meio de chaves substitutas, compondo o modelo estrela definido para o projeto.

Dessa forma, a implementação e a persistência da camada Gold foram concluídas com sucesso.

## 6.9 Conclusão da construção da camada Gold

A camada Gold foi construída a partir dos dados tratados e consolidados na camada Silver, implementando o modelo dimensional definido previamente para o projeto.

Foram criadas quatro dimensões — Tempo, Campo, Poço e Instalação — e uma tabela fato central de produção. As dimensões receberam chaves substitutas próprias, posteriormente utilizadas como chaves estrangeiras na tabela `fato_producao`.

A tabela fato manteve a granularidade de uma observação mensal por combinação de poço, campo e instalação e armazenou duas medidas preparadas para consumo analítico:

- `producao_oleo_mmbbl`: produção de óleo expressa em **milhões de barris (MMbbl)**;
- `producao_gas_bilhoes_m3`: produção total de gás natural expressa em **bilhões de metros cúbicos**.

As medidas foram derivadas dos valores preservados na camada Silver, mantendo os valores nulos existentes na fonte e realizando as conversões de unidade apenas para fins de consumo analítico.

As validações realizadas confirmaram:

- preservação dos **323.307 registros** da camada Silver;
- correspondência de todos os registros com as quatro dimensões;
- ausência de chaves estrangeiras nulas;
- ausência de combinações repetidas na granularidade definida;
- cobertura temporal de janeiro de 2005 a dezembro de 2025;
- preservação dos valores nulos existentes nas medidas de produção.

Ao final, o modelo dimensional foi persistido em formato Delta no schema `workspace.gold`, disponibilizando uma estrutura analítica organizada e com medidas em unidades adequadas para responder às perguntas de negócio definidas no projeto.